**Retention potvrdil, že 74,1 % zákazníků se vrací.
Teď chceme vědět: liší se tyto skupiny chováním?**


In [1]:
import pandas as pd
from scipy import stats

In [4]:
zakaznici=pd.read_csv("objednavky50.csv")

zakaznici.info()

<class 'pandas.DataFrame'>
RangeIndex: 14670 entries, 0 to 14669
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_objednavky   14670 non-null  int64  
 1   datum           14670 non-null  str    
 2   id_zakaznik     14670 non-null  int64  
 3   cena_celkem     14670 non-null  int64  
 4   typ_nakupu      14670 non-null  str    
 5   zpusob_platby   14670 non-null  str    
 6   id_kampane      2248 non-null   float64
 7   id_pobocky      14670 non-null  int64  
 8   pocet_nakupu    14670 non-null  int64  
 9   dny_v_datasetu  14670 non-null  int64  
 10  prvni_nakup     14670 non-null  str    
 11  typ_zakaznik    14670 non-null  str    
dtypes: float64(1), int64(6), str(5)
memory usage: 1.3 MB


**Agregace dat: počet nákupů a průměrná objednávka na zákazníka**


In [5]:
case = zakaznici.groupby('id_zakaznik').agg(
    pocet_nakupu=('id_objednavky', 'count'),
    prumerna_objednavka=('cena_celkem', 'mean')
).reset_index()
case.describe()


,id_zakaznik,pocet_nakupu,prumerna_objednavka
count,4864.000000,4864.000000,4864.000000
mean,3048.631579,3.016036,10321.184706
std,2015.329461,1.790184,8683.963072
min,1.000000,1.000000,200.000000
25%,1322.750000,2.000000,4500.000000
50%,2812.500000,3.000000,8195.000000
75%,4537.500000,4.000000,13600.000000
max,7280.000000,14.000000,70000.000000


**Segmentace zákazníků na základě percentilů**

In [6]:
case["typ_zakaznik"] = pd.cut(
    case["pocet_nakupu"], 
    bins=[0, 1, 3, 14], 
    labels=["jednorazovy", "prilezitostny", "verny"]
    )

case

,id_zakaznik,pocet_nakupu,prumerna_objednavka,typ_zakaznik
0,1,14,16257.142857,verny
1,2,12,5883.333333,verny
2,3,12,11633.333333,verny
3,4,11,7286.363636,verny
4,5,11,12550.000000,verny
...,...,...,...,...
4859,7272,1,4000.000000,jednorazovy
4860,7275,1,1900.000000,jednorazovy
4861,7276,1,2800.000000,jednorazovy
4862,7278,1,24000.000000,jednorazovy


**ANOVA: existuje statisticky významný rozdíl mezi segmenty?**

In [7]:
one = case[case["typ_zakaznik"] == "jednorazovy"]["prumerna_objednavka"]
two = case[case["typ_zakaznik"] == "prilezitostny"]["prumerna_objednavka"]
three = case[case["typ_zakaznik"] == "verny"]["prumerna_objednavka"]

stats.f_oneway(one, two, three)

F_onewayResult(statistic=np.float64(18.186623135710057), pvalue=np.float64(1.3522397651246664e-08))

**Závěr ANOVA**

P-hodnota: 1.35e-08 (prakticky nulová)

Rozdíl v průměrné hodnotě objednávky mezi zákaznickými segmenty 
je statisticky velmi významný. Pro konkrétní hodnoty jednotlivých 
segmentů viz následující analýza.

**Průměrná hodnota objednávky podle segmentu**

In [16]:
case.groupby("typ_zakaznik")["prumerna_objednavka"].mean()

typ_zakaznik
jednorazovy      11700.230840
prilezitostny     9802.695502
verny            10084.981303
Name: prumerna_objednavka, dtype: float64

**Závěr**

Ačkoliv byl potvrzen statisticky významný rozdíl mezi segmenty, průměrná hodnota objednávky je paradoxně nejvyšší u jednorázových zákazníků (11 700 Kč) a nejnižší u věrných (10 084 Kč). 

Toto zjištění otevírá další otázky - dalším krokem by byla například celková hodnota zákazníka v čase (LTV). 

